# Tarefa 1 — Coleta bruta (RIPE Atlas)

Coleta **somente com GET** na API pública do RIPE Atlas. Nenhuma medição é criada. Este notebook:

1. valida a medição 2016 e escolhe as outras 6 (ping IPv4, passo curto, destinos em países distintos);
2. escolhe 10 probes por medição, cada uma em um país de origem diferente;
3. baixa o **Período A** e o **Período B** (JSON original, intocado, em `raw/api_ripe/`);
4. limpa a tabela da medição 2016 e junta os 30 CSVs pequenos que já existiam;
5. grava as tabelas em `data/interim/` e imprime a qualidade da coleta.

Não há baseline, rótulo nem divisão treino/teste aqui (isso é a Tarefa 2). Todos os parâmetros ficam em `config/coleta_config.json`.

## 1. Configuração

In [12]:
import json, time, random, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import requests

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "config" / "coleta_config.json").exists())
CAMINHO_CFG = RAIZ / "config" / "coleta_config.json"
CFG = json.loads(CAMINHO_CFG.read_text(encoding="utf-8"))

API = CFG["api"]["base"]
DIR_JSON = RAIZ / CFG["caminhos"]["raw"] / "api_ripe"
DIR_INTERIM = RAIZ / CFG["caminhos"]["interim"]
DIR_INTERIM.mkdir(parents=True, exist_ok=True)

def epoch(iso):
    return int(pd.Timestamp(iso).timestamp())

PER = {k: (epoch(v["inicio"]), epoch(v["fim"])) for k, v in CFG["periodos"].items()}
assert PER["A"][1] == PER["B"][0], "A e B devem ser contíguos (fim de A = início de B)."
MIN_VALIDOS = CFG["rotulagem"]["min_rtt_validos"]
# 7 dias com 1.500 RTT válidos exigem passo de no máximo ~403 s.
LIMITE_PASSO_S = int((PER["A"][1] - PER["A"][0]) / MIN_VALIDOS)
print("Raiz do repositório:", RAIZ)
print("Passo máximo aceito (s):", LIMITE_PASSO_S)

S = requests.Session()
S.headers["User-Agent"] = "icmpmaster-aula/1.0 (projeto academico, somente GET)"

def get_json(url, params=None):
    """GET com repetição para erros transitórios. Nunca usa POST."""
    h, ultimo = CFG["api"], None
    for i in range(h["tentativas"]):
        try:
            r = S.get(url, params=params, timeout=h["timeout_s"])
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 500, 502, 503, 504):
                ultimo = f"HTTP {r.status_code}"
                ra = r.headers.get("Retry-After", "")
                time.sleep(float(ra) if ra.isdigit() else h["espera_s"] * 2 ** i)
                continue
            r.raise_for_status()
        except (requests.Timeout, requests.ConnectionError) as e:
            ultimo = repr(e)
            time.sleep(h["espera_s"] * 2 ** i)
    raise RuntimeError(f"Falha em {url}: {ultimo}")

def salvar_config():
    CAMINHO_CFG.write_text(json.dumps(CFG, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

Raiz do repositório: c:\Users\duduf\icmpmaster
Passo máximo aceito (s): 403


## 2. Medições

A medição 2016 já está definida. Para as medições sem `msm_id` no config, o notebook procura **pings IPv4 públicos da malha de Anchors** (um destino por Anchor) e escolhe uma por país de destino, só entre as que têm passo curto o bastante para chegar a 1.500 RTT válidos em 7 dias. A escolha é gravada no config e não muda nas execuções seguintes.

In [13]:
N = CFG["coleta"]["probes_por_medicao"]

def paises_das_ancoras():
    """Mapa IP/FQDN da Anchor -> país."""
    mapa, url, params = {}, f"{API}/anchors/", {"page_size": 100}
    while url:
        d = get_json(url, params); params = None
        for a in d.get("results", []):
            for k in ("ip_v4", "address_v4", "fqdn"):
                if a.get(k):
                    mapa[a[k]] = a.get("country")
        url = d.get("next")
    return mapa

def info_medicao(msm):
    m = get_json(f"{API}/measurements/{msm}/")
    return {"id": msm, "tipo": m.get("type"), "af": m.get("af"), "passo_s": m.get("interval"),
            "alvo": m.get("target_ip") or m.get("target"), "alvo_nome": m.get("target"),
            "inicio": m.get("start_time"), "publica": m.get("is_public")}

def candidatas(mapa, max_paginas=30):
    base = {"type": "ping", "af": 4, "is_public": "true", "status": 2, "page_size": 100}
    for extra in ({"tags": "anchoring-mesh"}, {}):
        url, params, achadas = f"{API}/measurements/", {**base, **extra}, []
        for _ in range(max_paginas):
            d = get_json(url, params); params = None
            for m in d.get("results", []):
                pais = mapa.get(m.get("target_ip")) or mapa.get(m.get("target"))
                if (pais and m.get("interval") and m["interval"] <= LIMITE_PASSO_S
                        and m.get("start_time") and m["start_time"] <= PER["A"][0]):
                    achadas.append({"id": m["id"], "pais": pais, "passo_s": m["interval"],
                                    "alvo": m.get("target_ip") or m.get("target")})
            url = d.get("next")
            if not url:
                break
        if achadas:
            return achadas
    return []

def probes_ativas(msm, t0):
    d = get_json(f"{API}/measurements/{msm}/results/",
                 {"start": t0, "stop": t0 + CFG["coleta"]["janela_amostra_s"], "format": "json"})
    return {r["prb_id"] for r in d if "prb_id" in r}

def paises_das_probes(ids):
    ids, out = sorted(ids), {}
    for i in range(0, len(ids), 100):
        lote = ids[i:i + 100]
        d = get_json(f"{API}/probes/", {"id__in": ",".join(map(str, lote)),
                                         "fields": "id,country_code", "page_size": 100})
        for p in d.get("results", []):
            out[p["id"]] = p.get("country_code")
    return out

ELEGIVEIS = {}   # msm_id -> {probe_id: país}, guardado só para esta sessão

def elegivel(msm):
    """Medição serve se tem N probes ativas no início de A e de B, em N países diferentes."""
    if msm in ELEGIVEIS:
        return ELEGIVEIS[msm]
    comuns = probes_ativas(msm, PER["A"][0]) & probes_ativas(msm, PER["B"][0])
    pa = paises_das_probes(comuns) if len(comuns) >= N else {}
    ok = len({c for c in pa.values() if c}) >= N
    ELEGIVEIS[msm] = pa if ok else None
    if not ok:
        print(f"  descartada: msm {msm} ({len(comuns)} probes em comum, {len(set(pa.values()))} países)")
    return ELEGIVEIS[msm]

faltam = [m for m in CFG["medicoes"] if m["msm_id"] is None]
if faltam:
    mapa_anc = paises_das_ancoras()
    usados = {m["dst_pais"] for m in CFG["medicoes"] if m["dst_pais"]}
    ids_ja = {m["msm_id"] for m in CFG["medicoes"] if m["msm_id"]}
    cands = [c for c in candidatas(mapa_anc) if c["id"] not in ids_ja]
    rng = random.Random(CFG["coleta"]["seed"])
    por_pais = {}
    for c in sorted(cands, key=lambda c: c["id"]):
        por_pais.setdefault(c["pais"], []).append(c)
    paises = sorted(p for p in por_pais if p not in usados)
    rng.shuffle(paises)
    print(f"{len(cands)} candidatas em {len(paises)} países; testando se têm {N} países de origem...")
    testadas = 0
    for slot in faltam:
        while paises and slot["msm_id"] is None:
            p = paises.pop(0)
            opcoes = por_pais[p][:]
            rng.shuffle(opcoes)
            for c in opcoes[:6]:
                testadas += 1
                if elegivel(c["id"]):
                    slot["msm_id"], slot["dst_pais"] = c["id"], p
                    print(f"  escolhida: msm {c['id']} (destino {p})")
                    break
        assert slot["msm_id"] is not None, (
            "Não achei medições suficientes com 10 países de origem. "
            "Preencha os msm_id à mão no config.")
    salvar_config()
    print(f"Medições gravadas em config/coleta_config.json ({testadas} candidatas testadas)")

linhas = []
for m in CFG["medicoes"]:
    i = info_medicao(m["msm_id"])
    assert i["tipo"] == "ping", f"Medição {m['msm_id']} não é ping."
    assert i["passo_s"] and i["passo_s"] <= LIMITE_PASSO_S, (
        f"Medição {m['msm_id']}: passo de {i['passo_s']} s não alcança {MIN_VALIDOS} RTT em 7 dias.")
    linhas.append({"msm_id": i["id"], "af": i["af"], "passo_s": i["passo_s"], "alvo": i["alvo"],
                   "dst_pais": m["dst_pais"]})
display(pd.DataFrame(linhas))
paises_dst = [m["dst_pais"] for m in CFG["medicoes"] if m["dst_pais"]]
assert len(paises_dst) == len(set(paises_dst)), "Há países de destino repetidos."

,msm_id,af,passo_s,alvo,dst_pais
0,2016,6,240,2001:503:c27::2:30,NaN
1,8937123,4,240,87.238.48.84,NO
2,19206759,4,240,143.255.248.149,EC
3,32089417,4,240,63.222.187.5,PH
4,23574809,4,240,103.77.105.3,ID
5,22680685,4,240,148.163.220.5,BR
6,29539835,4,240,193.1.33.11,IE


## 3. Probes

Para cada medição: 10 probes, **cada uma em um país de origem diferente**. Se o país do destino é conhecido e há probe nele, uma probe fica nesse país (caminho curto); as demais ficam em países distintos (caminho longo). A escolha usa só país e sorteio com seed fixa, nunca o RTT. Só entram probes que reportaram no início de A **e** no início de B.

In [14]:
def escolher_probes(paises, dst_pais, n, seed):
    rng = random.Random(seed)
    por_pais = {}
    for pid, c in sorted(paises.items()):
        if c:
            por_pais.setdefault(c, []).append(pid)
    esc, usados = [], set()
    if dst_pais in por_pais:
        esc.append(rng.choice(por_pais[dst_pais])); usados.add(dst_pais)
    outros = [c for c in sorted(por_pais) if c not in usados]
    rng.shuffle(outros)
    for c in outros:
        if len(esc) >= n:
            break
        esc.append(rng.choice(por_pais[c]))
    assert len(esc) == n, f"Só {len(esc)} países de origem distintos disponíveis (precisa de {n})."
    return sorted(esc)

for m in CFG["medicoes"]:
    if len(m["probes"]) == N:
        continue
    pa = elegivel(m["msm_id"])
    assert pa, (f"msm {m['msm_id']} não tem {N} probes em {N} países de origem diferentes. "
                "Troque o msm_id no config (ou ponha null para sortear outra).")
    m["probes"] = escolher_probes(pa, m["dst_pais"], N, CFG["coleta"]["seed"] + m["msm_id"])
    salvar_config()
    print(f"msm {m['msm_id']}: {len(pa)} probes elegíveis -> {m['probes']}")
todas = {p for m in CFG["medicoes"] for p in m["probes"]}

## 4. Baixar Períodos A e B (JSON original)

Um arquivo por medição, período e janela de `chunk_horas`. O conteúdo da API é gravado **sem transformação**. Arquivo que já existe é pulado, então a célula pode ser repetida. O fim de cada janela é exclusivo, então A e B não se sobrepõem.

In [15]:
def baixar(msm, probes, periodo):
    ini, fim = PER[periodo]
    passo = CFG["coleta"]["chunk_horas"] * 3600
    tag = hashlib.md5(",".join(map(str, sorted(probes))).encode()).hexdigest()[:8]
    pasta = DIR_JSON / f"msm{msm}"
    pasta.mkdir(parents=True, exist_ok=True)
    novos = pulados = 0
    t = ini
    while t < fim:
        t2 = min(t + passo, fim)
        nome = pasta / f"{periodo}_{pd.Timestamp(t, unit='s', tz='UTC'):%Y%m%dT%H%M}_p{tag}.json"
        if nome.exists():
            pulados += 1
        else:
            dados = get_json(f"{API}/measurements/{msm}/results/",
                             {"start": t, "stop": t2 - 1, "probe_ids": ",".join(map(str, probes)),
                              "format": "json"})
            tmp = nome.with_suffix(".tmp")
            tmp.write_text(json.dumps(dados, ensure_ascii=False), encoding="utf-8")
            tmp.replace(nome)
            novos += 1
        t = t2
    return novos, pulados

for m in CFG["medicoes"]:
    for per in ("A", "B"):
        n, p = baixar(m["msm_id"], m["probes"], per)
        print(f"msm {m['msm_id']} período {per}: {n} novos, {p} já existentes")

msm 2016 período A: 0 novos, 7 já existentes
msm 2016 período B: 0 novos, 7 já existentes
msm 8937123 período A: 0 novos, 7 já existentes
msm 8937123 período B: 0 novos, 7 já existentes
msm 19206759 período A: 0 novos, 7 já existentes
msm 19206759 período B: 0 novos, 7 já existentes
msm 32089417 período A: 0 novos, 7 já existentes
msm 32089417 período B: 0 novos, 7 já existentes
msm 23574809 período A: 0 novos, 7 já existentes
msm 23574809 período B: 0 novos, 7 já existentes
msm 22680685 período A: 0 novos, 7 já existentes
msm 22680685 período B: 0 novos, 7 já existentes
msm 29539835 período A: 0 novos, 7 já existentes
msm 29539835 período B: 0 novos, 7 já existentes


## 5. Do JSON à tabela tratada

Cada linha é uma rodada de ping. Regras (da Tarefa 1):

- RTT ausente fica **vazio**, nunca 0. O `-1` da API nunca entra: o RTT da rodada é a média dos RTT individuais com resposta (`rtt_medio_ms`).
- `perda_pct` = (enviados − recebidos) / enviados × 100. Com `enviados = 0` fica vazia.
- `jitter_ms` = desvio-padrão (amostral) dos RTT da rodada, só com 2 ou mais respostas.
- `timeout_atual` = 1 se não há RTT ou se `perda_pct` = 100.
- Duplicata = mesma `measurement_id`, `probe_id`, `dst_addr` e `timestamp`; fica a primeira.
- `fluxo_id` = `probe_id|dst_addr`.

In [16]:
COLS = ["timestamp", "measurement_id", "probe_id", "dst_addr", "fluxo_id", "rtt_medio_ms",
        "rtt_min_ms", "rtt_max_ms", "enviados", "recebidos", "perda_pct", "jitter_ms", "timeout_atual"]
CHAVE = ["measurement_id", "probe_id", "dst_addr", "timestamp"]

def parse_registro(r):
    res = r.get("result") or []
    rtts = [x["rtt"] for x in res if isinstance(x, dict) and x.get("rtt") is not None and not x.get("dup")]
    env = r.get("sent") if r.get("sent") is not None else len(res)
    rec = r.get("rcvd") if r.get("rcvd") is not None else len(rtts)
    return {
        "timestamp": r.get("timestamp"), "measurement_id": r.get("msm_id"), "probe_id": r.get("prb_id"),
        "dst_addr": r.get("dst_addr") or r.get("dst_name"),
        "rtt_medio_ms": float(np.mean(rtts)) if rtts else np.nan,
        "rtt_min_ms": float(min(rtts)) if rtts else np.nan,
        "rtt_max_ms": float(max(rtts)) if rtts else np.nan,
        "enviados": env, "recebidos": rec,
        "jitter_ms": float(np.std(rtts, ddof=1)) if len(rtts) >= 2 else np.nan,
    }

def de_json(registros):
    return pd.DataFrame([parse_registro(r) for r in registros])

def completar(df):
    """Regras comuns: perda, RTT <= 0, timeout, fluxo_id, datas em UTC."""
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True) \
        if np.issubdtype(df["timestamp"].dtype, np.number) else pd.to_datetime(df["timestamp"], utc=True)
    for c in ("rtt_medio_ms", "rtt_min_ms", "rtt_max_ms"):
        df.loc[df[c] <= 0, c] = np.nan
    if "perda_pct" not in df:
        df["perda_pct"] = np.nan
    calc = (df["enviados"] - df["recebidos"]) / df["enviados"].where(df["enviados"] > 0) * 100
    df["perda_pct"] = np.where(df["enviados"].notna(), calc, df["perda_pct"])
    df["timeout_atual"] = (df["rtt_medio_ms"].isna() | (df["perda_pct"] == 100)).astype(int)
    df["fluxo_id"] = df["probe_id"].astype(str) + "|" + df["dst_addr"].astype(str)
    return df[COLS]

def sem_duplicatas(df):
    n0 = len(df)
    df = df.drop_duplicates(CHAVE, keep="first").sort_values(["fluxo_id", "timestamp"]).reset_index(drop=True)
    return df, n0 - len(df)

# --- API (Períodos A e B)
partes = []
for arq in sorted(DIR_JSON.rglob("*.json")):
    reg = json.loads(arq.read_text(encoding="utf-8"))
    if reg:
        partes.append(de_json(reg))
api = completar(pd.concat(partes, ignore_index=True)) if partes else pd.DataFrame(columns=COLS)
print("Registros lidos da API:", len(api))

Registros lidos da API: 345770


### 5.1 Tabela da medição 2016 (coleta antiga de 1 hora)

O JSON antigo (`raw/probes_csv_1_measurement/`) é lido, limpo com as mesmas regras e gravado completo em `data/interim/medicao_2016_limpa.csv`. Na tabela final entram **só as probes escolhidas** para a 2016.

In [17]:
bruto_2016 = json.loads((RAIZ / CFG["caminhos"]["legado_2016_json"]).read_text(encoding="utf-8"))
t16 = completar(de_json(bruto_2016))
n_bruto_2016 = len(t16)
t16, dup_2016 = sem_duplicatas(t16)
aud16 = {
    "registros_brutos": n_bruto_2016,
    "duplicatas_removidas": dup_2016,
    "sondas": int(t16["probe_id"].nunique()),
    "sem_rtt (vazio)": int(t16["rtt_medio_ms"].isna().sum()),
    "timeout_atual=1": int(t16["timeout_atual"].sum()),
    "enviados=0 (perda vazia)": int((t16["enviados"] == 0).sum()),
    "jitter vazio (<2 respostas)": int(t16["jitter_ms"].isna().sum()),
    "janela (UTC)": f"{t16['timestamp'].min():%Y-%m-%d %H:%M} a {t16['timestamp'].max():%Y-%m-%d %H:%M}",
}
display(pd.Series(aud16, name="medição 2016").to_frame())
t16.to_csv(DIR_INTERIM / "medicao_2016_limpa.csv", index=False)
probes_2016 = next(m["probes"] for m in CFG["medicoes"] if m["msm_id"] == 2016)
t16_sel = t16[t16["probe_id"].isin(probes_2016)]
print("Linhas da 2016 nas probes escolhidas:", len(t16_sel))

,medição 2016
registros_brutos,109882
duplicatas_removidas,0
sondas,7494
sem_rtt (vazio),10807
timeout_atual=1,10807
enviados=0 (perda vazia),267
jitter vazio (<2 respostas),10898
janela (UTC),2026-09-15 14:42 a 2026-09-15 15:42


Linhas da 2016 nas probes escolhidas: 145


### 5.2 Os 30 CSVs pequenos (5 medições × 6 probes)

Entram no mesmo formato. Esses CSVs não têm `enviados`, `recebidos` nem as rodadas individuais, então ficam vazios: `enviados`, `recebidos`, `rtt_min_ms`, `rtt_max_ms` e `jitter_ms` (o jitter antigo era outra fórmula, a média das diferenças entre RTT consecutivos, e não pode ser misturado ao desvio-padrão). Eles cobrem 1 hora de 09/10 e ficam **fora dos Períodos A e B**.

In [18]:
arqs = sorted((RAIZ / CFG["caminhos"]["legado_5_csvs"]).glob("*.csv"))
d5 = pd.concat([pd.read_csv(a) for a in arqs], ignore_index=True)
c5 = pd.DataFrame({
    "timestamp": d5["timestamp"], "measurement_id": d5["msm_id"], "probe_id": d5["prb_id"],
    "dst_addr": d5["destino"], "rtt_medio_ms": d5["ping_ms"], "rtt_min_ms": np.nan, "rtt_max_ms": np.nan,
    "enviados": np.nan, "recebidos": np.nan, "perda_pct": d5["perda_pacotes_pct"], "jitter_ms": np.nan,
})
c5 = completar(c5)
n_bruto_c5 = len(c5)
c5, dup_c5 = sem_duplicatas(c5)
print(f"CSVs pequenos: {len(arqs)} arquivos, {n_bruto_c5} linhas, {dup_c5} duplicatas, "
      f"{c5['fluxo_id'].nunique()} fluxos")

CSVs pequenos: 30 arquivos, 799 linhas, 0 duplicatas, 30 fluxos


## 6. Tabela final e tabela de países

In [19]:
n_bruto_api = len(api)
tab = pd.concat([api, t16_sel, c5], ignore_index=True)
n_antes = len(tab)
tab, dup_total = sem_duplicatas(tab)
tab.to_csv(DIR_INTERIM / "medicoes.csv", index=False)
print("Linhas gravadas em data/interim/medicoes.csv:", len(tab), "| duplicatas removidas:", dup_total)

# País de origem e de destino de cada fluxo (só para auditar diversidade; não entra nas variáveis do modelo).
dst_pais = {m["msm_id"]: m["dst_pais"] for m in CFG["medicoes"]}
fluxos = tab[["measurement_id", "probe_id", "dst_addr", "fluxo_id"]].drop_duplicates().copy()
fluxos["pais_origem"] = fluxos["probe_id"].map(paises_das_probes(set(fluxos["probe_id"])))
fluxos["pais_destino"] = fluxos["measurement_id"].map(dst_pais)
fluxos["caminho"] = np.where(fluxos["pais_destino"].isna(), "indeterminado",
                     np.where(fluxos["pais_origem"] == fluxos["pais_destino"], "curto", "longo"))
fluxos.sort_values(["measurement_id", "probe_id"]).to_csv(DIR_INTERIM / "probes_paises.csv", index=False)
display(fluxos.groupby("measurement_id").agg(probes=("probe_id", "nunique"),
        paises_origem=("pais_origem", "nunique"), pais_destino=("pais_destino", "first")))

Linhas gravadas em data/interim/medicoes.csv: 346569 | duplicatas removidas: 145


,probes,paises_origem,pais_destino
measurement_id,,,
2011,6,5,NaN
2012,6,5,NaN
2013,6,5,NaN
2015,6,5,NaN
2016,16,13,NaN
8937123,10,10,NO
19206759,10,10,EC
22680685,10,10,BR
23574809,10,10,ID


## 7. Qualidade da coleta (sem baseline, sem rótulo)

Aqui só se descreve o que foi coletado. Nenhuma linha recebe OK, RISCO ou FALHA.

In [20]:
seg = (tab["timestamp"] - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta(seconds=1)
tab["periodo"] = np.select(
    [(seg >= PER["A"][0]) & (seg < PER["A"][1]), (seg >= PER["B"][0]) & (seg < PER["B"][1])],
    ["A", "B"], "fora")

print("Linhas por período:"); display(tab["periodo"].value_counts().rename("linhas").to_frame())
por_fluxo = tab.groupby("fluxo_id").agg(
    registros=("timestamp", "size"), inicio=("timestamp", "min"), fim=("timestamp", "max"),
    rtt_vazio=("rtt_medio_ms", lambda s: int(s.isna().sum())),
    timeouts=("timeout_atual", "sum"))
display(por_fluxo.describe().T)

print("\nRTT (ms) e perda, descritivos (sem classe):")
display(pd.DataFrame({
    "rtt_medio_ms": tab["rtt_medio_ms"].describe(),
    "perda_pct": tab["perda_pct"].describe()}))

assert (tab["rtt_medio_ms"].dropna() > 0).all(), "RTT <= 0 encontrado."
a_b = tab[tab["periodo"].isin(["A", "B"])]
print("Fluxos com dados em A:", a_b[a_b.periodo == "A"]["fluxo_id"].nunique(),
      "| em B:", a_b[a_b.periodo == "B"]["fluxo_id"].nunique())
print("Caminhos presentes:", fluxos["caminho"].value_counts().to_dict())

Linhas por período:


,linhas
periodo,
A,173676
B,172094
fora,799


,count,mean,std,min,25%,50%,75%,max
registros,100.0,3465.69,2290.733506,23.0,28.0,5038.0,5040.0,5040.0
rtt_vazio,100.0,216.40,951.248335,0.0,0.0,1.0,7.5,5040.0
timeouts,100.0,216.40,951.248335,0.0,0.0,1.0,7.5,5040.0



RTT (ms) e perda, descritivos (sem classe):


,rtt_medio_ms,perda_pct
count,324929.000000,346501.000000
mean,116.165053,6.437788
std,96.853086,24.319984
min,0.070134,0.000000
25%,38.594867,0.000000
50%,83.482360,0.000000
75%,198.017039,0.000000
max,2418.282038,100.000000


Fluxos com dados em A: 70 | em B: 70
Caminhos presentes: {'longo': 54, 'indeterminado': 40, 'curto': 6}


## 8. Números para o diário

In [21]:
print("N de registros brutos (API + 2016 nas probes escolhidas + CSVs pequenos):",
      n_bruto_api + len(t16_sel) + n_bruto_c5)
print("Registros após remover duplicatas (tabela tratada):", len(tab))
print("Duplicatas removidas na junção:", dup_total)
print("N de fluxos:", tab["fluxo_id"].nunique())
print("Fluxos com dados nos períodos A/B:", a_b["fluxo_id"].nunique(), "| só fora dos períodos:",
      tab["fluxo_id"].nunique() - a_b["fluxo_id"].nunique())
print("Caminho curto e longo presentes:", sorted(set(fluxos["caminho"]) - {"indeterminado"}))
print("Registros com RTT vazio:", int(tab["rtt_medio_ms"].isna().sum()),
      "| timeouts (timeout_atual=1):", int(tab["timeout_atual"].sum()))
print("Medições:", [m["msm_id"] for m in CFG["medicoes"]])
print("Períodos:", CFG["periodos"])

N de registros brutos (API + 2016 nas probes escolhidas + CSVs pequenos): 346714
Registros após remover duplicatas (tabela tratada): 346569
Duplicatas removidas na junção: 145
N de fluxos: 100
Fluxos com dados nos períodos A/B: 70 | só fora dos períodos: 30
Caminho curto e longo presentes: ['curto', 'longo']
Registros com RTT vazio: 21640 | timeouts (timeout_atual=1): 21640
Medições: [2016, 8937123, 19206759, 32089417, 23574809, 22680685, 29539835]
Períodos: {'A': {'inicio': '2026-09-06T04:32:00Z', 'fim': '2026-09-13T04:32:00Z'}, 'B': {'inicio': '2026-09-13T04:32:00Z', 'fim': '2026-09-20T04:32:00Z'}}


In [22]:
pp = pd.read_csv(DIR_INTERIM / "probes_paises.csv")
print(pp["caminho"].value_counts())
cols = ["measurement_id", "probe_id", "pais_origem", "pais_destino"]
print("\nCurtos:")
display(pp[pp["caminho"] == "curto"][cols])
print("\nUm longo por medição:")
display(pp[pp["caminho"] == "longo"].groupby("measurement_id").head(1)[cols])

caminho
longo            54
indeterminado    40
curto             6
Name: count, dtype: int64

Curtos:


,measurement_id,probe_id,pais_origem,pais_destino
49,8937123,7406,NO,NO
59,19206759,1016538,EC,EC
64,22680685,7113,BR,BR
79,23574809,1010583,ID,ID
89,29539835,1016347,IE,IE
90,32089417,7032,PH,PH



Um longo por medição:


,measurement_id,probe_id,pais_origem,pais_destino
40,8937123,6380,GH,NO
50,19206759,12849,VE,EC
60,22680685,6358,UG,BR
70,23574809,18087,GR,ID
80,29539835,13347,NO,IE
91,32089417,17846,IL,PH
